In [66]:
%pip install --upgrade timm torch torchvision
%pip install timm==0.9.12

import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import timm
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

  Using cached timm-1.0.24-py3-none-any.whl.metadata (38 kB)
Using cached timm-1.0.24-py3-none-any.whl (2.6 MB)
  Attempting uninstall: timm
    Found existing installation: timm 0.9.12
    Uninstalling timm-0.9.12:
      Successfully uninstalled timm-0.9.12
Note: you may need to restart the kernel to use updated packages.
  Using cached timm-0.9.12-py3-none-any.whl.metadata (60 kB)
Using cached timm-0.9.12-py3-none-any.whl (2.2 MB)
  Attempting uninstall: timm
    Found existing installation: timm 1.0.24
    Uninstalling timm-1.0.24:
      Successfully uninstalled timm-1.0.24
Note: you may need to restart the kernel to use updated packages.


In [75]:
import torch
import torch.nn as nn
import torch.optim as optim

import time
import math
import random
import numpy as np

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [77]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [78]:
cvt_models=timm.list_models("cvt*")
print(cvt_models)


[]


In [79]:
cvt_models[:20]

[]

In [80]:
len(cvt_models)

0

In [81]:
def pick_model_name(names):
    prefs=['cvt_13','cvt-13','cvt13','cvt_21','cvt-21','cvt21']
    for p in prefs:
        for n in names:
            if n.startswith(p):
                return n
    return names[0] if names else None

model_name = pick_model_name(cvt_models)

if model_name is None:
    print('Warning: No CvT model found. Using a default model.')
    model_name = 'cvt_13'

print(f'Selected model: {model_name}')

Selected model: cvt_13


In [82]:
IMG_SIZE=224
BATCH_SIZE=64 
NUM_WORKERS=2
EPOCHS=5

train_tfms=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [83]:
test_tfms=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914,0.4812,0.4465),std=(0.2470,0.2435,0.2616))
])

In [84]:
data_root="./data"
train_ds=datasets.CIFAR10(root=data_root,train=True,download=True,transform=train_tfms)
test_ds=datasets.CIFAR10(root=data_root,train=True,download=True,transform=test_tfms)

train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
len(train_ds)
len(test_ds)

Files already downloaded and verified
Files already downloaded and verified


50000

In [85]:
len(test_ds)

50000

In [86]:
# Use a standard Vision Transformer model since CvT models are not available
model_name = 'vit_tiny_patch16_224'

model = timm.create_model(model_name, pretrained=True, num_classes=10)
model = model.to(device)
x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    y = model(x)
    print(f'Model output shape: {y.shape}')

Model output shape: torch.Size([2, 10])


In [87]:
y.shape

torch.Size([2, 10])

In [89]:
def accuracy_top1(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion, epoch=0):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    for step, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        running_acc += accuracy_top1(logits.detach(), labels)
    return running_loss / len(loader), running_acc / len(loader)

In [90]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        running_loss += loss.item()
        running_acc += accuracy_top1(logits, labels)
    return running_loss / len(loader), running_acc / len(loader)

In [ ]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-2)
best_acc=0.0
for  epoch in range(3):
    t0=time.time()  
    tr_loss,tr_acc=train_one_epoch(model,train_loader,optimizer,criterion,epoch=epoch)
    te_loss,te_acc=evaluate(model,test_loader,criterion)
    best_acc=max(best_acc,te_acc)
    dt=time.time()-t0
    print(f"Epoch[{epoch+1}/3], train_loss:{tr_loss:.4f}, train_acc:{tr_acc:.4f}, test_loss:{te_loss:.4f}, test_acc:{te_acc:.4f}, best_acc:{best_acc:.4f}, time:{dt:.2f}s") 
    print("Best Test Acc: ", best_acc)

Epoch[1/3], train_loss:1.5995, train_acc:0.4072, test_loss:1.3500, test_acc:0.5163, best_acc:0.5163, time:2704.21s
Best Test Acc:  0.5162643861892583


In [ ]:
classes=train_ds.classes
@torch.no_grad()
def predict_image(model, image):
    model.eval()
    image = test_tfms(image).unsqueeze(0).to(device)
    logits = model(image)
    probs = torch.softmax(logits, dim=1)
    conf, pred = torch.max(probs, dim=1)
    return classes[pred.item()], conf.item()    
img_path="./data/CIFAR10/test/cat.4000.jpg"
image=Image.open(img_path).convert("RGB")
pred_class, pred_conf=predict_image(model,image)
print(f"Predicted Class: {pred_class}, Confidence: {pred_conf:.4f}")    